# Food Availability Research Pipeline
## Literature NLP → Cross-Country Empirical Modelling (Colab / Jupyter)

This single notebook replicates the complete dissertation pipeline:

| Phase | Task |
|-------|------|
| A4 | Score literature corpus for strict availability-side alignment |
| A3 | NLP topic modelling — TF-IDF, LDA coherence sweep, NMF fallback |
| B  | Download all country-level data (World Bank WDI, FAO, Findex, IMF, WGI) |
| C  | Build and clean master cross-country dataset + feature engineering |
| D  | FAO Food Balance Sheet DV + OLS / RF / XGBoost models |
| F  | Nested F-test synthesis — does NLP identify empirically predictive themes? |

**Approximate runtime:** 15–25 min in Colab (most time in Phase B downloads and Phase D bootstrap).

> **To use in Google Colab:** upload your `data/raw/corpus_metadata.csv` to Drive (or mount this repo), then run all cells top-to-bottom.

In [ ]:
# ── Install packages not pre-installed in Colab ────────────────────────
import sys
import subprocess
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'gensim>=4.0', 'xgboost', 'shap', 'pycountry', 'statsmodels'],
    check=False
)

# ── Standard library ────────────────────────────────────────────────────
import io, os, re, time, warnings, zipfile, datetime
warnings.filterwarnings('ignore')

# ── Third-party ─────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import requests
import pycountry
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.decomposition import NMF
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import KNNImputer
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import shap
from scipy.stats import f as f_dist

print('All packages loaded OK')


## Setup: Working Directory

Run the cell below to mount Google Drive (Colab) or set a local path.  
All outputs are written relative to `BASE_DIR`.

In [ ]:
# ── Detect Colab ──────────────────────────────────────────────────────
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        # Change this path to wherever you uploaded the project folder in Drive
        BASE_DIR = '/content/drive/MyDrive/Research_Food_Insecurity_Project'
        print('Google Drive mounted. BASE_DIR =', BASE_DIR)
    except Exception as e:
        print('Drive mount failed:', e)
        BASE_DIR = '/content/Research_Food_Insecurity_Project'
        print('Using local Colab storage:', BASE_DIR)
else:
    # Running locally — use the current working directory
    BASE_DIR = os.getcwd()
    print('Local Jupyter. BASE_DIR =', BASE_DIR)

os.chdir(BASE_DIR)

# ── Create all output directories ─────────────────────────────────────
for d in ['data/raw', 'data/processed', 'outputs/tables',
          'outputs/figures', 'outputs/narrative']:
    os.makedirs(d, exist_ok=True)

print('Output directories ready.')
RANDOM_SEED = 42


---
## Phase A4 — Score Literature Corpus

Reads `data/raw/corpus_metadata.csv` (papers from OpenAlex + Scopus + manual PDFs) and scores each paper against the dissertation themes.  
Papers must have:
- A food-security core term (food security, food availability, hunger…)
- At least one explicit **availability-side** term (cereal production, post-harvest loss, grain storage…)

Output → `data/processed/strictly_aligned_papers.csv` (127 papers in the project run)

In [ ]:
# ── Term dictionaries ─────────────────────────────────────────────────
CORE_TERMS = {
    'food security': 4, 'food insecurity': 4,
    'food availability': 5, 'cereal availability': 5,
    'food supply': 4, 'food supplies': 4,
    'dietary energy supply': 4,
    'hunger': 3, 'undernourishment': 2, 'undernourished': 2,
    'malnutrition': 1, 'food access': 1, 'nutrition security': 1,
}

FOOD_SECURITY_CORE_TERMS = [
    'food security', 'food insecurity', 'food availability',
    'cereal availability', 'hunger',
]

AVAILABILITY_TERMS = [
    # Availability / supply measurement
    'food availability', 'cereal availability', 'food supply', 'food supplies',
    'dietary energy supply', 'domestic supply', 'calorie availability',
    'caloric availability', 'calorie supply', 'caloric supply',
    'available food', 'food balance sheet', 'food balance',
    'grain supply', 'cereal supply',
    # Aggregate production
    'food production', 'food output', 'agricultural output', 'cereal output',
    'crop production', 'cereal production', 'agricultural production',
    'grain production', 'staple crop', 'staple crops',
    'cereal grain', 'cereal grains',
    # Crop-specific
    'rice production', 'wheat production', 'maize production',
    'sorghum production', 'millet production', 'barley production',
    # Yield
    'cereal yield', 'crop yield', 'yield gap',
    'rice yield', 'wheat yield', 'maize yield',
    # Post-harvest / storage
    'post-harvest loss', 'postharvest loss',
    'post-harvest losses', 'postharvest losses',
    'food loss', 'food losses', 'storage loss',
    'grain storage', 'cold chain',
    # Supply-chain / logistics
    'value chain', 'supply chain', 'food value chain', 'food system',
]

THEMES = {
    'post_harvest_loss':         ['post-harvest', 'postharvest', 'food loss', 'food losses',
                                   'food waste', 'storage loss', 'grain storage', 'cold chain'],
    'value_chain_market_access': ['value chain', 'supply chain', 'market access',
                                   'marketing channel', 'logistics'],
    'financial_access':          ['financial access', 'financial inclusion', 'credit access',
                                   'agricultural credit', 'rural finance', 'microfinance',
                                   'mobile money'],
    'smallholder_agriculture':   ['smallholder', 'smallholders', 'farm household',
                                   'rural household', 'farmer', 'farmers'],
    'production_yield_cereals':  ['cereal', 'cereals', 'maize', 'wheat', 'rice',
                                   'yield', 'productivity', 'fertilizer', 'fertiliser',
                                   'irrigation'],
    'climate_environment':       ['climate change', 'climate variability', 'rainfall',
                                   'drought', 'water scarcity', 'resilience'],
    'governance_institutions':   ['governance', 'institutional', 'political stability',
                                   'corruption', 'policy'],
    'gender_poverty_inclusion':  ['women', 'female', 'gender', 'poorest', 'poverty'],
    'empirical_methods':         ['regression', 'cross-country', 'random forest',
                                   'machine learning', 'panel data', 'household survey'],
}

AVAILABILITY_DRIVER_THEMES = [
    'post_harvest_loss', 'value_chain_market_access', 'production_yield_cereals'
]
PROJECT_DRIVER_THEMES = [
    'post_harvest_loss', 'value_chain_market_access', 'financial_access',
    'smallholder_agriculture', 'climate_environment', 'governance_institutions'
]

SKIP_TERMS = [
    'genome', 'molecular hydrogen', 'microplasma', 'far-uvc',
    'fungal contamination', 'mycotoxin accumulation', 'pathogen',
    'aspergillus', 'fusarium', 'transcriptome', 'rna splicing',
    'virulence', 'protein source', 'meat analogue', 'robotic',
    'culinary applications', 'lateral flow immunoassay',
    'aflatoxin b1 detection', 'mineral content',
]

print('Term dictionaries defined.')


In [ ]:
# ── Scoring functions ──────────────────────────────────────────────────

def clean_text(value):
    return '' if pd.isna(value) else str(value).lower()

def contains_phrase(text, phrase):
    escaped = re.escape(phrase.lower())
    pattern = r'(?<![a-z0-9])' + escaped + r'(?![a-z0-9])'
    return re.search(pattern, text) is not None

def match_terms(text, terms):
    return [t for t in terms if contains_phrase(text, t)]

def score_one_paper(row):
    title    = clean_text(row.get('title', ''))
    abstract = clean_text(row.get('abstract', ''))
    full     = title + ' ' + abstract

    skip_matches         = match_terms(full, SKIP_TERMS)
    availability_matches = match_terms(full, AVAILABILITY_TERMS)
    title_avail          = match_terms(title, AVAILABILITY_TERMS)

    score = 0
    core_matches = []
    title_core   = []
    fs_core      = []

    for term, weight in CORE_TERMS.items():
        if contains_phrase(title, term):
            score += weight + 2
            core_matches.append(term); title_core.append(term)
            if term in FOOD_SECURITY_CORE_TERMS:
                fs_core.append(term)
        elif contains_phrase(abstract, term):
            score += weight
            core_matches.append(term)
            if term in FOOD_SECURITY_CORE_TERMS:
                fs_core.append(term)

    if availability_matches:
        score += 3 + len(title_avail)

    theme_matches = []
    for theme_name, terms in THEMES.items():
        matched = match_terms(full, terms)
        if matched:
            theme_matches.append(theme_name)
            score += 2
        score += len(match_terms(title, terms))

    if clean_text(row.get('source_db', '')) == 'scopus': score += 1
    if len(abstract) >= 500: score += 1
    try:
        if int(float(row.get('year', 0))) >= 2015: score += 1
    except Exception:
        pass
    if skip_matches: score -= 5

    avail_driver = sum(1 for t in theme_matches if t in AVAILABILITY_DRIVER_THEMES)
    proj_driver  = sum(1 for t in theme_matches if t in PROJECT_DRIVER_THEMES)
    is_pdf       = clean_text(row.get('source_db', '')) == 'pdf'
    has_fs_core  = bool(fs_core)
    has_avail    = bool(availability_matches)

    if is_pdf:
        if score >= 7 and has_fs_core and has_avail and not skip_matches:
            level = 'strict'
        elif score >= 7 and proj_driver >= 2 and has_avail and not skip_matches:
            level = 'strict'
        elif score >= 4 and not skip_matches and (core_matches or theme_matches):
            level = 'moderate'
        else:
            level = 'weak'
    else:
        if (score >= 12 and has_fs_core and has_avail and not skip_matches
                and (title_core or title_avail or avail_driver >= 2)):
            level = 'strict'
        elif score >= 7 and core_matches and theme_matches and not skip_matches:
            level = 'moderate'
        else:
            level = 'weak'

    return pd.Series({
        'alignment_score':               score,
        'alignment_level':               level,
        'matched_core_terms':            '; '.join(sorted(set(core_matches))),
        'matched_availability_terms':    '; '.join(sorted(set(availability_matches))),
        'matched_themes':                '; '.join(sorted(set(theme_matches))),
        'theme_count':                   len(theme_matches),
        'project_driver_theme_count':    proj_driver,
        'availability_driver_theme_count': avail_driver,
        'skip_terms':                    '; '.join(sorted(set(skip_matches))),
    })

print('Scoring functions defined.')


In [ ]:
# ── Run Phase A4 ──────────────────────────────────────────────────────

corpus = pd.read_csv('data/raw/corpus_metadata.csv')
print(f'Corpus loaded: {len(corpus)} papers')

scores  = corpus.apply(score_one_paper, axis=1)
scored  = pd.concat([corpus, scores], axis=1)

level_order = {'strict': 1, 'moderate': 2, 'weak': 3}
scored['level_order'] = scored['alignment_level'].map(level_order)
scored = (scored
          .sort_values(['level_order', 'alignment_score', 'year'],
                       ascending=[True, False, False])
          .drop(columns=['level_order']))

strict = scored[scored['alignment_level'] == 'strict'].copy()

summary = (
    scored.groupby(['alignment_level', 'source_db'])
    .size().reset_index(name='papers')
    .sort_values(['alignment_level', 'source_db'])
)

scored.to_csv('data/processed/scored_literature_alignment.csv', index=False)
strict.to_csv('data/processed/strictly_aligned_papers.csv', index=False)
summary.to_csv('outputs/tables/literature_alignment_summary.csv', index=False)

print(f'Strict papers: {len(strict)} (target >= 100)')
print('Source breakdown:')
print(strict['source_db'].value_counts().to_string())
print(summary.to_string(index=False))


---
## Phase A3 — NLP Topic Modelling

Runs on the 127 strictly-aligned papers.  
**Two-track approach:**
1. **LDA** with coherence sweep K=3–12. Target c_v ≥ 0.60. If threshold is met, LDA topics are used directly.
2. **TF-IDF + NMF** is always run as the fallback (and as the reported result when LDA falls short of the target). NMF produces directly interpretable availability-side topics.

Outputs:
- `data/processed/tfidf_top_keywords.csv`
- `data/processed/nmf_availability_topics.csv`
- `outputs/figures/lda_coherence_curve.png`

In [ ]:
# ── NLP helper: domain stop-words ─────────────────────────────────────

DOMAIN_STOP = {
    'food', 'security', 'insecurity', 'study', 'analysis', 'result',
    'results', 'paper', 'using', 'used', 'data', 'model', 'models',
    'countries', 'country', 'based', 'also', 'level', 'levels',
    'among', 'within', 'between', 'across', 'higher', 'lower',
    'significant', 'significantly', 'associated', 'use', 'effect',
    'effects', 'factor', 'factors', 'increase', 'decrease',
    'related', 'high', 'low', 'index', 'area', 'household',
    'households', 'rural', 'urban', 'region', 'regional',
    'world', 'global', 'national', 'local', 'year', 'years',
    'provide', 'order', 'set', 'type', 'different', 'large',
    'small', 'new', 'total', 'rate', 'rates', 'number',
    'sample', 'variable', 'variables',
}

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
ALL_STOP = DOMAIN_STOP | set(ENGLISH_STOP_WORDS)

print('Stop-word set size:', len(ALL_STOP))


In [ ]:
# ── TF-IDF + NMF ──────────────────────────────────────────────────────
import gensim
from gensim import corpora
from gensim.models import LdaMulticore, CoherenceModel, Phrases
from gensim.models.phrases import Phraser

strict_df = pd.read_csv('data/processed/strictly_aligned_papers.csv')
print(f'Strict papers loaded: {len(strict_df)}')

# Build text field from title + abstract
strict_df['text'] = (
    strict_df['title'].fillna('') + ' ' + strict_df['abstract'].fillna('')
).str.lower()

# ── TF-IDF ────────────────────────────────────────────────────────────
tfidf_vec = TfidfVectorizer(
    max_features=500,
    ngram_range=(1, 2),
    stop_words=list(ALL_STOP),
    min_df=2,
    max_df=0.85,
)
tfidf_matrix = tfidf_vec.fit_transform(strict_df['text'])
feature_names = tfidf_vec.get_feature_names_out()

# Corpus-wide top keywords
mean_tfidf = np.asarray(tfidf_matrix.mean(axis=0)).flatten()
top_idx    = mean_tfidf.argsort()[::-1][:30]
tfidf_rows = []
for rank, idx in enumerate(top_idx, 1):
    tfidf_rows.append({'scope': 'corpus-wide', 'rank': rank,
                       'keyword': feature_names[idx],
                       'mean_tfidf': round(mean_tfidf[idx], 5)})
tfidf_df = pd.DataFrame(tfidf_rows)
tfidf_df.to_csv('data/processed/tfidf_top_keywords.csv', index=False)
print('Top 15 corpus-wide TF-IDF keywords:')
print(', '.join(tfidf_df.head(15)['keyword'].tolist()))

# ── NMF ───────────────────────────────────────────────────────────────
nmf_k = min(8, max(3, len(strict_df) // 15))
nmf_model = NMF(n_components=nmf_k, init='nndsvda',
                random_state=RANDOM_SEED, max_iter=1000)
W = nmf_model.fit_transform(tfidf_matrix)
H = nmf_model.components_

nmf_rows = []
for topic_id in range(nmf_k):
    top_feat_idx = H[topic_id].argsort()[::-1][:12]
    top_kws      = ', '.join(feature_names[top_feat_idx[:8]])
    dominant     = int((W.argmax(axis=1) == topic_id).sum())
    nmf_rows.append({'topic_id': topic_id, 'top_keywords': top_kws,
                     'n_dominant_docs': dominant})
    print(f'  NMF Topic {topic_id}: {top_kws}  ({dominant} papers)')

nmf_df = pd.DataFrame(nmf_rows)
nmf_df.to_csv('data/processed/nmf_availability_topics.csv', index=False)
print('NMF topics saved.')


In [ ]:
# ── LDA coherence sweep K=3..12 ────────────────────────────────────────

# Tokenise and build bigrams
tokenised = [
    [w for w in re.split(r'[^a-z]+', text) if len(w) > 2 and w not in ALL_STOP]
    for text in strict_df['text']
]

bigram_model  = Phrases(tokenised, min_count=3, threshold=8)
bigram_phraser = Phraser(bigram_model)
tokenised_bi  = [bigram_phraser[doc] for doc in tokenised]

dictionary = corpora.Dictionary(tokenised_bi)
dictionary.filter_extremes(no_below=2, no_above=0.75)
corpus_bow = [dictionary.doc2bow(doc) for doc in tokenised_bi]
print(f'Dictionary size: {len(dictionary)} unique tokens')

coherence_scores = {}
K_RANGE = range(3, 13)

for k in K_RANGE:
    lda_k = LdaMulticore(
        corpus_bow, num_topics=k, id2word=dictionary,
        passes=20, workers=1, random_state=RANDOM_SEED,
    )
    cm = CoherenceModel(
        model=lda_k, texts=tokenised_bi,
        dictionary=dictionary, coherence='c_v'
    )
    score = cm.get_coherence()
    coherence_scores[k] = round(score, 4)
    print(f'  K={k}: c_v = {score:.4f}')

best_k       = max(coherence_scores, key=coherence_scores.get)
best_score   = coherence_scores[best_k]
LDA_THRESHOLD = 0.60
print(f'Best K={best_k}, coherence={best_score:.4f} (threshold={LDA_THRESHOLD})')

# ── Coherence curve chart ──────────────────────────────────────────────
plt.figure(figsize=(8, 4))
plt.plot(list(coherence_scores.keys()), list(coherence_scores.values()),
         marker='o')
plt.axhline(LDA_THRESHOLD, linestyle='--', color='red', label='Target 0.60')
plt.xlabel('Number of topics K')
plt.ylabel('LDA Coherence (c_v)')
plt.title('LDA Coherence Sweep — Strictly Aligned Papers')
plt.legend()
plt.tight_layout()
plt.savefig('outputs/figures/lda_coherence_curve.png', dpi=150)
plt.close()
print('Coherence curve saved → outputs/figures/lda_coherence_curve.png')

if best_score >= LDA_THRESHOLD:
    print(f'LDA meets target (>= {LDA_THRESHOLD}). Fitting final model K={best_k}...')
    lda_final = LdaMulticore(
        corpus_bow, num_topics=best_k, id2word=dictionary,
        passes=40, workers=1, random_state=RANDOM_SEED,
        alpha='symmetric', eta='auto', per_word_topics=True,
    )
    print('LDA final model fitted.')
else:
    lda_final = None
    print(f'LDA below threshold ({best_score:.4f} < {LDA_THRESHOLD}). '
          f'Using TF-IDF + NMF as primary result.')

# Save theme→variable mapping hint
theme_map = pd.DataFrame([
    {'nlp_theme': 'Post-harvest loss / grain storage',
     'proxy_variable': 'cereal_loss_pct', 'source': 'APHLIS + FAO FBS'},
    {'nlp_theme': 'Market integration / value chain',
     'proxy_variable': 'trade_pct_gdp', 'source': 'WDI NE.TRD.GNFS.ZS'},
    {'nlp_theme': 'Technology / storage infrastructure',
     'proxy_variable': 'rural_electricity_access_pct', 'source': 'WDI EG.ELC.ACCS.RU.ZS'},
    {'nlp_theme': 'Land productivity / input efficiency',
     'proxy_variable': 'fertiliser_efficiency', 'source': 'cereal_yield / fertiliser_kg_per_ha'},
    {'nlp_theme': 'Climate / market disruption signal',
     'proxy_variable': 'food_price_inflation_pct', 'source': 'WDI FP.CPI.TOTL.ZG'},
])
theme_map.to_csv('data/processed/phase_A_theme_variable_mapping.csv', index=False)
print('Theme→variable mapping saved.')


---
## Phase B — Download Country-Level Data

Downloads all predictor variables from public APIs:  
- **World Bank WDI** — yield, fertiliser, land, GDP, population, employment, etc.
- **World Bank Findex** (2021) — financial inclusion indicators
- **IMF Financial Access Survey** — bank branches, ATMs, credit
- **WGI Governance Indicators** (source=3)
- **Trade % of GDP** — logistics / market-integration proxy (NE.TRD.GNFS.ZS)
- **PHL** — post-harvest loss (from cached `data/raw/phl_combined.csv` if present,
  otherwise constructs a regional proxy)

> Each WB call downloads ~350 countries and is filtered to real sovereign states. Expect ~2–5 minutes for this section.


In [ ]:
# ── World Bank regional / aggregate codes to exclude ──────────────────
REGIONAL_CODES = {
    'AFE','AFW','ARB','CEB','CSS','EAP','EAR','EAS','ECA','ECS',
    'EMU','EUU','FCS','HIC','HPC','IBD','IBT','IDA','IDB','IDX',
    'LAC','LCN','LDC','LIC','LMC','LMY','LTE','MEA','MIC','MNA',
    'NAC','OED','OSS','PRE','PSS','PST','SAS','SSA','SSF','SST',
    'TEA','TEC','TLA','TMN','TSA','TSS','UMC','WLD','XZN',
}

def download_wb(indicator, column_name, year=2021, source=None):
    """Download one World Bank WDI indicator for all countries."""
    url = f'https://api.worldbank.org/v2/country/all/indicator/{indicator}'
    params = {'date': year, 'format': 'json', 'per_page': 400}
    if source is not None:
        params['source'] = source
    try:
        resp  = requests.get(url, params=params, timeout=30)
        data  = resp.json()
        if len(data) < 2 or data[1] is None:
            return None
        rows = [
            {'country_code': e.get('countryiso3code', ''),
             'country_name': e.get('country', {}).get('value', ''),
             column_name: e['value'],
             'year': year}
            for e in data[1]
            if e.get('value') is not None
            and e.get('countryiso3code') not in REGIONAL_CODES
            and len(e.get('countryiso3code', '')) == 3
        ]
        return pd.DataFrame(rows) if rows else None
    except Exception as ex:
        print(f'  WB download failed ({indicator}): {ex}')
        return None

print('WB download helper defined.')


In [ ]:
# ── WDI core agricultural + economic indicators ────────────────────────
WDI_INDICATORS = [
    ('AG.YLD.CREL.KG',   'cereal_yield_kg_per_ha'),
    ('AG.CON.FERT.ZS',   'fertiliser_kg_per_ha'),
    ('AG.LND.ARBL.ZS',   'arable_land_pct'),
    ('AG.LND.IRIG.AG.ZS','irrigation_pct'),
    ('AG.PRD.LVSK.XD',   'livestock_production_index'),
    ('NY.GDP.PCAP.KD',   'gdp_per_capita_usd'),
    ('NV.AGR.TOTL.ZS',   'agri_value_added_pct_gdp'),
    ('FP.CPI.TOTL.ZG',   'food_price_inflation_pct'),
    ('SP.POP.TOTL',       'population_total'),
    ('SP.RUR.TOTL.ZS',   'rural_population_pct'),
    ('SL.AGR.EMPL.ZS',   'agri_employment_pct'),
    ('SL.AGR.EMPL.FE.ZS','female_agri_employment_pct'),
    ('IT.NET.USER.ZS',   'internet_users_pct'),
    ('EG.ELC.ACCS.RU.ZS','rural_electricity_access_pct'),
    ('SH.STA.STNT.ZS',   'child_stunting_pct'),
    ('IC.LGL.CRED.XQ',   'credit_rights_index'),
    ('NE.TRD.GNFS.ZS',   'trade_pct_gdp'),
]

wdi_frames = []
for indicator, col in WDI_INDICATORS:
    df_i = download_wb(indicator, col)
    if df_i is not None and len(df_i) > 0:
        wdi_frames.append(df_i[['country_code', 'country_name', col]])
        print(f'  {col:<42} {len(df_i):>3} countries')
    else:
        print(f'  {col:<42} FAILED')
    time.sleep(0.3)

# Merge all WDI indicators on country_code
if wdi_frames:
    wdi_merged = wdi_frames[0]
    for df_i in wdi_frames[1:]:
        col = [c for c in df_i.columns if c not in ('country_code', 'country_name')][0]
        wdi_merged = wdi_merged.merge(
            df_i[['country_code', col]], on='country_code', how='outer'
        )
    wdi_merged = wdi_merged[~wdi_merged['country_code'].isin(REGIONAL_CODES)]
    wdi_merged.to_csv('data/raw/worldbank_wdi_2021.csv', index=False)
    print(f'WDI saved: {len(wdi_merged)} countries, {wdi_merged.shape[1]} columns')
else:
    print('No WDI data downloaded — check network connection')
    wdi_merged = pd.DataFrame()


In [ ]:
# ── Findex (financial inclusion) ────────────────────────────────────────
FINDEX_INDICATORS = [
    ('FX.OWN.TOTL.ZS',   'account_ownership_pct'),
    ('FX.OWN.TOTL.RU.ZS','account_ownership_rural_pct'),
    ('FX.OWN.TOTL.FE.ZS','account_ownership_female_pct'),
    ('FX.OWN.TOTL.PL.ZS','account_ownership_poorest40_pct'),
    ('FX.TRN.FINM.ZS',   'financial_mobile_pct'),
    ('FX.TRN.AGRI.ZS',   'agri_payments_digital_pct'),
]

findex_frames = []
for indicator, col in FINDEX_INDICATORS:
    # Findex 2021 = source 71 in WB API
    df_i = download_wb(indicator, col, year=2021, source=71)
    if df_i is not None and len(df_i) > 0:
        findex_frames.append(df_i[['country_code', col]])
        print(f'  {col:<42} {len(df_i):>3} countries')
    time.sleep(0.3)

# Borrow from bank (not in standard source)
borrow_df = download_wb('FD.AST.PRVT.GD.ZS', 'borrowed_from_bank_pct')
if borrow_df is not None:
    findex_frames.append(borrow_df[['country_code', 'borrowed_from_bank_pct']])

if findex_frames:
    findex_merged = findex_frames[0]
    for df_i in findex_frames[1:]:
        col = [c for c in df_i.columns if c != 'country_code'][0]
        findex_merged = findex_merged.merge(df_i, on='country_code', how='outer')
    findex_merged.to_csv('data/raw/findex_2021.csv', index=False)
    print(f'Findex saved: {len(findex_merged)} countries')

# ── IMF Banking (branches, ATMs, credit) ─────────────────────────────────
IMF_INDICATORS = [
    ('FB.CBK.BRCH.P5', 'bank_branches_per_100k'),
    ('FB.ATM.TOTL.P5', 'atm_per_100k'),
    ('FS.AST.PRVT.GD.ZS', 'private_credit_pct_gdp'),
]
imf_frames = []
for indicator, col in IMF_INDICATORS:
    df_i = download_wb(indicator, col)
    if df_i is not None and len(df_i) > 0:
        imf_frames.append(df_i[['country_code', col]])
        print(f'  {col:<42} {len(df_i):>3} countries')
    time.sleep(0.3)
if imf_frames:
    imf_merged = imf_frames[0]
    for df_i in imf_frames[1:]:
        imf_merged = imf_merged.merge(df_i, on='country_code', how='outer')
    imf_merged.to_csv('data/raw/imf_financial_access.csv', index=False)
    print(f'IMF saved: {len(imf_merged)} countries')


In [ ]:
# ── WGI Governance Indicators (source=3) ──────────────────────────────
WGI_INDICATORS = [
    ('CC.EST',  'wgi_control_of_corruption'),
    ('GE.EST',  'wgi_government_effectiveness'),
    ('PV.EST',  'wgi_political_stability'),
    ('RQ.EST',  'wgi_regulatory_quality'),
    ('RL.EST',  'wgi_rule_of_law'),
    ('VA.EST',  'wgi_voice_accountability'),
]
wgi_frames = []
for indicator, col in WGI_INDICATORS:
    df_i = download_wb(indicator, col, source=3)
    if df_i is not None and len(df_i) > 0:
        wgi_frames.append(df_i[['country_code', col]])
        print(f'  {col:<42} {len(df_i):>3} countries')
    time.sleep(0.3)
if wgi_frames:
    wgi_merged = wgi_frames[0]
    for df_i in wgi_frames[1:]:
        wgi_merged = wgi_merged.merge(df_i, on='country_code', how='outer')
    wgi_merged.to_csv('data/raw/wgi_governance_2021.csv', index=False)
    print(f'WGI saved: {len(wgi_merged)} countries')
else:
    print('WGI download failed — models will skip WGI predictors')


In [ ]:
# ── Post-Harvest Loss — use existing file or construct proxy ──────────

PHL_FILE = 'data/raw/phl_combined.csv'

if os.path.exists(PHL_FILE):
    phl_df = pd.read_csv(PHL_FILE)
    print(f'PHL file loaded: {len(phl_df)} countries')
else:
    # Build a simple sub-regional proxy from FAO FBS loss element
    # Element 5123 = Losses, 5301 = Domestic supply quantity
    print('phl_combined.csv not found — building regional proxy...')
    SUB_REGIONAL_PROXY = {
        'SSA': 14.5, 'SAS': 10.0, 'EAP': 9.0, 'MNA': 8.5,
        'LAC': 10.5, 'ECA': 6.0, 'NAC': 4.5, 'default': 10.0
    }
    # Map WB regions → proxy values
    try:
        region_resp = requests.get(
            'https://api.worldbank.org/v2/country?format=json&per_page=400',
            timeout=30
        )
        rdata = region_resp.json()[1]
        phl_rows = []
        for c in rdata:
            iso = c.get('id', '')
            reg = c.get('region', {}).get('id', 'default')
            if len(iso) == 3 and iso not in REGIONAL_CODES:
                phl_rows.append({'country_code': iso,
                                  'cereal_loss_pct': SUB_REGIONAL_PROXY.get(
                                      reg, SUB_REGIONAL_PROXY['default'])})
        phl_df = pd.DataFrame(phl_rows)
        phl_df.to_csv(PHL_FILE, index=False)
        print(f'PHL proxy saved: {len(phl_df)} countries (sub-regional proxies)')
    except Exception as ex:
        print(f'PHL proxy construction failed: {ex}')
        phl_df = pd.DataFrame(columns=['country_code', 'cereal_loss_pct'])


---
## Phase C — Build Master Cross-Country Dataset

Merges all downloaded files into a single master table (one row per country) and engineers two composite variables:
- `fertiliser_efficiency` = cereal yield / fertiliser kg/ha
- `value_chain_finance_score` = average of Findex value-chain-specific components

Output → `data/processed/master_dataset_clean.csv`

In [ ]:
# ── Load and merge all data sources ──────────────────────────────────
print('Building master dataset...')

wdi_raw = pd.read_csv('data/raw/worldbank_wdi_2021.csv')
master  = wdi_raw[~wdi_raw['country_code'].isin(REGIONAL_CODES)].drop_duplicates(
    subset='country_code', keep='first'
).reset_index(drop=True)
print(f'Base WDI: {len(master)} countries, {master.shape[1]} columns')

def _merge(master, path, key='country_code'):
    if not os.path.exists(path):
        print(f'  SKIP (not found): {path}')
        return master
    raw = pd.read_csv(path)
    raw = raw.drop_duplicates(subset=key, keep='first')
    drop = [c for c in ['country_name', 'year'] if c in raw.columns]
    raw = raw.drop(columns=drop, errors='ignore')
    merged = master.merge(raw, on=key, how='left')
    print(f'  Merged {path}: shape {merged.shape}')
    return merged

master = _merge(master, 'data/raw/findex_2021.csv')
master = _merge(master, 'data/raw/imf_financial_access.csv')
master = _merge(master, 'data/raw/wgi_governance_2021.csv')
master = _merge(master, 'data/raw/phl_combined.csv')

# PHL proxy fallback (if phl_combined not merged yet)
if 'cereal_loss_pct' not in master.columns and len(phl_df) > 0:
    master = master.merge(
        phl_df[['country_code', 'cereal_loss_pct']], on='country_code', how='left'
    )

print(f'Master after all merges: {len(master)} countries, {master.shape[1]} cols')


In [ ]:
# ── Feature engineering ───────────────────────────────────────────────
print('Engineering derived features...')

# 1. Fertiliser efficiency
if 'cereal_yield_kg_per_ha' in master.columns and 'fertiliser_kg_per_ha' in master.columns:
    fert_nz = master['fertiliser_kg_per_ha'].replace(0, np.nan)
    master['fertiliser_efficiency'] = (master['cereal_yield_kg_per_ha'] / fert_nz).round(2)
    print(f'  fertiliser_efficiency: {master["fertiliser_efficiency"].notna().sum()} countries')

# 2. Value-chain finance composite score
VCHAIN_COLS = []
for col in ['account_ownership_rural_pct', 'agri_payments_digital_pct',
            'borrowed_from_bank_pct', 'account_ownership_female_pct',
            'account_ownership_poorest40_pct']:
    if col in master.columns and master[col].notna().sum() >= 50:
        VCHAIN_COLS.append(col)
for col, raw_col in [('bank_branches_scaled', 'bank_branches_per_100k'),
                      ('atm_scaled', 'atm_per_100k'),
                      ('credit_scaled', 'private_credit_pct_gdp')]:
    if raw_col in master.columns:
        col_max = master[raw_col].max()
        if pd.notna(col_max) and col_max > 0:
            master[col] = (master[raw_col] / col_max * 100).round(2)
            VCHAIN_COLS.append(col)

if VCHAIN_COLS:
    master['value_chain_finance_score'] = master[VCHAIN_COLS].mean(axis=1).round(2)
    print(f'  value_chain_finance_score from {VCHAIN_COLS}')
    print(f'  {master["value_chain_finance_score"].notna().sum()} countries have a score')

# ── Save ─────────────────────────────────────────────────────────────
master.to_csv('data/processed/master_dataset.csv', index=False)

CORE_COLS = ['cereal_yield_kg_per_ha', 'gdp_per_capita_usd', 'arable_land_pct']
core_mask = master[CORE_COLS].notna().all(axis=1)
master_clean = master[core_mask].copy().reset_index(drop=True)
master_clean.to_csv('data/processed/master_dataset_clean.csv', index=False)
print(f'master_dataset_clean.csv: {len(master_clean)} countries (all CORE_COLS present)')


---
## Phase D — Empirical Modelling

**Dependent variable:** cereal food supply per capita (kg/person/year) from the FAO Food Balance Sheet.  
This measures food actually *available* after production, imports, exports, and stock adjustments — not just domestic production.

**Download strategy (three attempts):**
1. FAOSTAT REST API (Item 2905, Element 664)
2. FAOSTAT bulk ZIP download (~55 MB) with M49→ISO3 conversion via `pycountry`
3. World Bank cereal production per capita fallback

**Models fitted:**
| Model | Predictors | Purpose |
|-------|-----------|--------|
| A | Production baseline (7 vars) | How much do physical inputs explain availability? |
| B | +Post-harvest loss | Does loss between farm and market reduce availability? |
| C | +Logistics/infrastructure | Does market integration / rural electricity matter? |
| F | NLP-discovered themes (12 vars) | Do themes the literature emphasises predict outcomes? |
| A★ | Baseline on Model F sample | Honest incremental comparison (same N) |


In [ ]:
# ── Download FAO FBS cereal food supply ───────────────────────────────
FAOSTAT_ITEM = '2905'   # Cereals - Excluding Beer
FAOSTAT_ELEM = '664'    # Food supply quantity (g/capita/day)
TARGET_YEAR  = 2021

dv_df = None

# Attempt 1 — FAOSTAT REST API
for base_url in [
    'https://www.fao.org/faostat/api/v1',
    'https://fenixservices.fao.org/faostat/api/v1',
]:
    try:
        resp = requests.get(
            f'{base_url}/en/data/FBS',
            params={'items': FAOSTAT_ITEM, 'elements': FAOSTAT_ELEM,
                    'years': TARGET_YEAR, 'area': '*', 'output_type': 'objects'},
            timeout=60,
        )
        if resp.status_code == 200:
            records = resp.json().get('data', [])
            if records:
                rows = []
                for rec in records:
                    iso3 = rec.get('Area Code (ISO3)', rec.get('area_code_iso3', ''))
                    val  = rec.get('Value', rec.get('value'))
                    if iso3 and val is not None and len(str(iso3)) == 3:
                        rows.append({'country_code': iso3,
                                     'cereal_availability_kg_pc': round(float(val)*365/1000, 2)})
                if rows:
                    dv_df = pd.DataFrame(rows)
                    print(f'  FAO API OK ({base_url}): {len(dv_df)} countries')
                    break
    except Exception as ex:
        print(f'  FAO API failed: {ex}')

# Attempt 2 — FAOSTAT bulk ZIP
if dv_df is None:
    for burl in [
        'https://bulks-faostat.fao.org/production/FoodBalanceSheets_E_All_Data_(Normalized).zip',
        'https://www.fao.org/faostat/static/bulkdownloads/FoodBalanceSheets_E_All_Data_(Normalized).zip',
    ]:
        try:
            print(f'  Trying bulk ZIP: {burl}')
            resp = requests.get(burl, timeout=180, allow_redirects=True, stream=True)
            if resp.status_code == 200:
                chunks, size = [], 0
                for chunk in resp.iter_content(chunk_size=1024*1024):
                    chunks.append(chunk)
                    size += len(chunk)
                    if size > 250*1024*1024:
                        break
                zf = zipfile.ZipFile(io.BytesIO(b''.join(chunks)))
                data_csv = next(n for n in zf.namelist()
                                if n.endswith('.csv') and 'AreaCode' not in n)
                fbs_raw = pd.read_csv(zf.open(data_csv), encoding='latin-1')
                fbs_raw.columns = [c.strip() for c in fbs_raw.columns]
                mask = (
                    (fbs_raw['Item Code'].astype(str) == FAOSTAT_ITEM)
                    & (fbs_raw['Element Code'].astype(str) == FAOSTAT_ELEM)
                    & (fbs_raw['Year'].astype(str) == str(TARGET_YEAR))
                )
                fbs_filt = fbs_raw[mask].copy()
                if len(fbs_filt) > 0:
                    def m49_to_iso3(m49_raw):
                        try:
                            num_str = str(m49_raw).lstrip("'").strip().zfill(3)
                            c = pycountry.countries.get(numeric=num_str)
                            return c.alpha_3 if c else None
                        except Exception:
                            return None
                    fbs_filt['country_code'] = fbs_filt['Area Code (M49)'].apply(m49_to_iso3)
                    fbs_filt = fbs_filt.dropna(subset=['country_code'])
                    fbs_filt['cereal_availability_kg_pc'] = (
                        pd.to_numeric(fbs_filt['Value'], errors='coerce') * 365 / 1000
                    )
                    dv_df = (fbs_filt[['country_code', 'cereal_availability_kg_pc']]
                             .dropna().reset_index(drop=True))
                    print(f'  Bulk ZIP OK: {len(dv_df)} countries')
                    break
        except Exception as ex:
            print(f'  Bulk ZIP failed: {ex}')

# Attempt 3 — World Bank production per capita fallback
if dv_df is None:
    print('  Using WB cereal production fallback...')
    prod_df = download_wb('AG.PRD.CREL.MT', 'prod_mt')
    pop_df  = download_wb('SP.POP.TOTL',    'pop')
    if prod_df is not None and pop_df is not None:
        merged = prod_df.merge(pop_df[['country_code', 'pop']], on='country_code')
        merged['cereal_availability_kg_pc'] = (merged['prod_mt'] * 1000 / merged['pop']).round(2)
        dv_df = merged[merged['cereal_availability_kg_pc'] >= 5][
            ['country_code', 'cereal_availability_kg_pc']
        ].reset_index(drop=True)
        print(f'  WB fallback: {len(dv_df)} countries (>= 5 kg/pc)')

dv_df['country_code'] = dv_df['country_code'].astype(str).str.strip()
dv_df.to_csv('data/raw/cereal_availability_2021.csv', index=False)
print(f'DV range: {dv_df["cereal_availability_kg_pc"].min():.1f} – '
      f'{dv_df["cereal_availability_kg_pc"].max():.1f} kg/person/year')


In [ ]:
# ── Model specifications ──────────────────────────────────────────────
DV = 'cereal_availability_kg_pc'

MODEL_A_VARS = [
    'cereal_yield_kg_per_ha',
    'fertiliser_kg_per_ha',
    'arable_land_pct',
    'gdp_per_capita_usd',
    'rural_population_pct',
    'agri_employment_pct',
    'livestock_production_index',
]

MODEL_B_VARS = MODEL_A_VARS + ['cereal_loss_pct']

MODEL_C_VARS = MODEL_B_VARS + [
    'trade_pct_gdp',
    'rural_electricity_access_pct',
]

# Model F — NLP-discovered availability themes
# Each predictor maps to a topic from the NMF analysis:
#   cereal_loss_pct           ← Topic 2 (post-harvest loss, grain storage)
#   trade_pct_gdp             ← Topic 6 (value chain, market integration)
#   rural_electricity_access_pct ← Topic 5 (technology, storage infrastructure)
#   fertiliser_efficiency     ← Topic 0 (land productivity, input efficiency)
#   food_price_inflation_pct  ← Topic 3 (climate / market disruption signal)
MODEL_F_VARS = MODEL_A_VARS + [
    'cereal_loss_pct',
    'trade_pct_gdp',
    'rural_electricity_access_pct',
    'fertiliser_efficiency',
    'food_price_inflation_pct',
]

MODELS = {
    'Model A — Baseline Production':       MODEL_A_VARS,
    'Model B — +Post-Harvest Loss':        MODEL_B_VARS,
    'Model C — +Logistics Infrastructure': MODEL_C_VARS,
    'Model F — NLP-Discovered Themes':     MODEL_F_VARS,
}

MIN_ML_N = 80
print('Model specs defined.')


In [ ]:
# ── Helper functions ──────────────────────────────────────────────────

LOG_COLS = [
    DV, 'gdp_per_capita_usd', 'cereal_yield_kg_per_ha',
    'population_total', 'fertiliser_kg_per_ha', 'fertiliser_efficiency',
    'trade_pct_gdp', 'bank_branches_per_100k', 'atm_per_100k',
    'private_credit_pct_gdp',
]

def prepare_data(df, predictor_cols, outcome_col):
    work = df.copy()
    if outcome_col == DV and DV in work.columns:
        work[DV] = np.log1p(work[DV].clip(lower=0))
    for col in LOG_COLS:
        if col in predictor_cols and col in work.columns:
            work[col] = np.log1p(work[col].clip(lower=0))

    existing = [c for c in predictor_cols if c in work.columns]
    needed   = [outcome_col] + existing

    # KNN impute columns with <= 20% missing
    to_impute = [c for c in existing
                 if 0 < work[c].isna().sum() / len(work) <= 0.20]
    if to_impute:
        imp = KNNImputer(n_neighbors=5)
        imp_arr = imp.fit_transform(work[existing])
        imp_df  = pd.DataFrame(imp_arr, columns=existing, index=work.index)
        for c in to_impute:
            n_filled = work[c].isna().sum()
            work[c]  = imp_df[c]
            print(f'    KNN-imputed {c}: {n_filled} gaps filled')

    work = work.dropna(subset=[c for c in needed if c in work.columns])
    X    = work[[c for c in existing if c in work.columns]].copy()
    y    = work[outcome_col].copy()
    return X, y, list(X.columns)


def run_ols(X, y, model_name, predictor_list):
    X_const = sm.add_constant(X)
    model   = sm.OLS(y, X_const).fit(cov_type='HC3')
    print(f'  OLS {model_name}: N={int(model.nobs)}, '
          f'R2={model.rsquared:.3f}, AdjR2={model.rsquared_adj:.3f}, '
          f'F-p={model.f_pvalue:.4f}')
    safe = (model_name.replace(' ', '_').replace('+', 'Plus')
                      .replace('\u2014', '').strip('_'))
    path = f'outputs/tables/ols_{safe}.txt'
    with open(path, 'w') as fh:
        fh.write(model.summary().as_text())
    return model


kfold = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

def run_ml(X, y, model_name):
    rf = RandomForestRegressor(
        n_estimators=200, max_depth=4, min_samples_leaf=3,
        random_state=RANDOM_SEED, n_jobs=-1
    )
    rf_cv  = cross_val_score(rf, X, y, cv=kfold, scoring='r2').mean()
    rf.fit(X, y)

    xgb_m = xgb.XGBRegressor(
        n_estimators=200, max_depth=3, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        random_state=RANDOM_SEED, verbosity=0
    )
    xgb_cv = cross_val_score(xgb_m, X, y, cv=kfold, scoring='r2').mean()
    print(f'  RF CV R2={rf_cv:.3f}  XGB CV R2={xgb_cv:.3f}')
    return rf, xgb_m, rf_cv, xgb_cv

print('Helper functions defined.')


In [ ]:
# ── Run all models ────────────────────────────────────────────────────

master_clean = pd.read_csv('data/processed/master_dataset_clean.csv')
master_dv    = master_clean.merge(dv_df, on='country_code', how='left')
master_dv.to_csv('data/processed/master_dataset_with_dv.csv', index=False)
print(f'Master+DV: {len(master_dv)} countries, '
      f'{master_dv[DV].notna().sum()} with DV data')

results = []

for model_name, predictor_cols in MODELS.items():
    print(f'\n{"="*60}')
    print(f'  {model_name}')
    print('='*60)

    X, y, used = prepare_data(master_dv, predictor_cols, DV)
    missing = [c for c in predictor_cols if c not in X.columns]
    if missing:
        print(f'  Missing vars (skipped): {missing}')
    print(f'  Complete countries: {len(X)}')

    if len(X) < 30:
        print('  Skipping — fewer than 30 countries')
        continue

    ols_m = run_ols(X, y, model_name, used)

    if len(X) >= MIN_ML_N:
        rf_m, xgb_m, rf_cv, xgb_cv = run_ml(X, y, model_name)
    else:
        rf_m = RandomForestRegressor(
            n_estimators=200, max_depth=4, min_samples_leaf=3,
            random_state=RANDOM_SEED, n_jobs=-1
        ).fit(X, y)
        rf_cv = xgb_cv = np.nan
        print(f'  N={len(X)} < {MIN_ML_N}: skipping CV')

    results.append({
        'Model':           model_name,
        'N (countries)':   len(X),
        'Predictors used': len(used),
        'OLS R²':           round(ols_m.rsquared, 3),
        'OLS Adj R²':       round(ols_m.rsquared_adj, 3),
        'OLS F-stat p':    round(ols_m.f_pvalue, 4),
        'RF 5-fold CV R²': round(rf_cv, 3) if not (isinstance(rf_cv, float) and np.isnan(rf_cv)) else np.nan,
        'XGB 5-fold CV R²':round(xgb_cv, 3) if not (isinstance(xgb_cv, float) and np.isnan(xgb_cv)) else np.nan,
    })

# ── Model A★ — same sample as F (honest comparison) ──────────────────
print('\nFitting Model A* (same sample as Model F)...')
X_f, y_f, _ = prepare_data(master_dv, MODEL_F_VARS, DV)
nlp_sample   = master_dv.loc[X_f.index].copy()
X_as, y_as, used_as = prepare_data(nlp_sample, MODEL_A_VARS, DV)

if len(X_as) >= 30:
    ols_as = run_ols(X_as, y_as, 'Model A★ — NLP sample', used_as)
    rf_as  = RandomForestRegressor(
        n_estimators=200, max_depth=4, min_samples_leaf=3,
        random_state=RANDOM_SEED, n_jobs=-1
    ).fit(X_as, y_as)
    results.append({
        'Model':           'Model A★ — NLP sample',
        'N (countries)':   len(X_as),
        'Predictors used': len(used_as),
        'OLS R²':           round(ols_as.rsquared, 3),
        'OLS Adj R²':       round(ols_as.rsquared_adj, 3),
        'OLS F-stat p':    round(ols_as.f_pvalue, 4),
        'RF 5-fold CV R²': np.nan,
        'XGB 5-fold CV R²':np.nan,
    })

results_df = pd.DataFrame(results)
results_df.to_csv('outputs/tables/model_comparison.csv', index=False)
print('\nModel comparison:')
print(results_df[['Model','N (countries)','OLS R²','OLS Adj R²']].to_string(index=False))


In [ ]:
# ── Bootstrap CIs (1000 iterations) ────────────────────────────────────
print('Running 1000-iteration bootstrap CIs...')

N_BOOT = 1000

def bootstrap_coefs(df, predictor_cols, outcome_col, n_iter=N_BOOT):
    X_all, y_all, used = prepare_data(df, predictor_cols, outcome_col)
    if len(X_all) < 30:
        return pd.DataFrame()
    store = {c: [] for c in used}
    rng = np.random.default_rng(RANDOM_SEED)
    for _ in range(n_iter):
        idx = rng.choice(len(X_all), len(X_all), replace=True)
        Xb, yb = X_all.iloc[idx], y_all.iloc[idx]
        try:
            m = sm.OLS(yb, sm.add_constant(Xb)).fit()
            for c in used:
                if c in m.params:
                    store[c].append(m.params[c])
        except Exception:
            pass
    rows = []
    for c, vals in store.items():
        if len(vals) >= 50:
            rows.append({
                'variable':    c,
                'boot_mean':   round(float(np.mean(vals)), 4),
                'ci_lower_95': round(float(np.percentile(vals, 2.5)), 4),
                'ci_upper_95': round(float(np.percentile(vals, 97.5)), 4),
                'n_valid':     len(vals),
            })
    return pd.DataFrame(rows)

boot_a = bootstrap_coefs(master_dv, MODEL_A_VARS, DV)
boot_a['model'] = 'Model A'
print('  Model A bootstrap done.')

boot_f = bootstrap_coefs(master_dv, MODEL_F_VARS, DV)
boot_f['model'] = 'Model F'
print('  Model F bootstrap done.')

boot_all = pd.concat([boot_a, boot_f], ignore_index=True)
boot_all.to_csv('outputs/tables/bootstrap_confidence_intervals.csv', index=False)

print('\nBootstrap 95% CIs — NLP predictors (Model F):')
nlp_vars = ['cereal_loss_pct','trade_pct_gdp','rural_electricity_access_pct',
            'fertiliser_efficiency','food_price_inflation_pct']
for _, row in boot_f[boot_f['variable'].isin(nlp_vars)].iterrows():
    excludes = row['ci_lower_95'] * row['ci_upper_95'] > 0
    status = 'EXCLUDES zero' if excludes else 'crosses zero'
    print(f'  {row["variable"]:<38} [{row["ci_lower_95"]:+.3f}, {row["ci_upper_95"]:+.3f}]  {status}')


In [ ]:
# ── Charts ─────────────────────────────────────────────────────────────

# R2 progression bar chart
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(results_df))
w = 0.25
rf_h  = results_df['RF 5-fold CV R²'].fillna(0)
xgb_h = results_df['XGB 5-fold CV R²'].fillna(0)
ax.bar(x - w, results_df['OLS R²'],  w, label='OLS R²',  color='#4472C4')
ax.bar(x,      rf_h,                 w, label='RF CV R²', color='#ED7D31')
ax.bar(x + w,  xgb_h,               w, label='XGB CV R²',color='#70AD47')
ax.set_xticks(x)
ax.set_xticklabels(
    [n.replace(' — ', '\n') for n in results_df['Model']], fontsize=8
)
ax.set_ylabel('R²')
ax.set_title('Model Performance Progression')
ax.legend()
plt.tight_layout()
plt.savefig('outputs/figures/model_r2_comparison.png', dpi=150)
plt.close()
print('R² chart saved.')

# Model F coefficient chart
X_fc, y_fc, _ = prepare_data(master_dv, MODEL_F_VARS, DV)
if len(X_fc) >= 30:
    ols_fc  = sm.OLS(y_fc, sm.add_constant(X_fc)).fit(cov_type='HC3')
    coefs   = ols_fc.params.drop('const')
    conf    = ols_fc.conf_int().drop('const')
    pvals   = ols_fc.pvalues.drop('const')
    order   = coefs.abs().sort_values().index
    nlp_hl  = set(MODEL_F_VARS) - set(MODEL_A_VARS)

    colours = []
    for v in order:
        p = pvals[v]
        colours.append('#C00000' if p<0.01 else '#FF6600' if p<0.05
                       else '#FFC000' if p<0.10 else '#AAAAAA')

    fig, ax = plt.subplots(figsize=(10, max(5, len(coefs)*0.45)))
    bars = ax.barh(range(len(order)), coefs[order].values, color=colours, alpha=0.85)
    for i, var in enumerate(order):
        if var in nlp_hl:
            bars[i].set_edgecolor('#1565C0'); bars[i].set_linewidth(2)
        ax.plot([conf.loc[var,0], conf.loc[var,1]], [i,i], color='black', lw=1.5)
    ax.axvline(0, color='black', lw=1)
    ax.set_yticks(range(len(order)))
    ax.set_yticklabels(list(order), fontsize=8)
    ax.set_title('Model F — NLP-Discovered Themes (blue border = NLP-discovered)')
    plt.tight_layout()
    plt.savefig('outputs/figures/model_f_nlp_coefficients.png', dpi=150)
    plt.close()
    print('Model F coefficient chart saved.')


---
## Phase F — Synthesis: Does NLP Predict Empirical Patterns?

Computes the **nested F-test** comparing Model A★ (baseline on the NLP sample) against Model F (baseline + 5 NLP-discovered predictors):  

$$F = \frac{(R^2_{\text{full}} - R^2_{\text{restricted}})/q}{(1 - R^2_{\text{full}})/(n - k_{\text{full}} - 1)}$$

A significant F means the NLP-discovered variables add **genuine predictive power** beyond the production baseline.  
Writes a plain-text narrative to `outputs/narrative/phase_f_synthesis.txt`.

In [ ]:
# ── Nested F-test ─────────────────────────────────────────────────────

models_csv = pd.read_csv('outputs/tables/model_comparison.csv')

row_a_star = models_csv[models_csv['Model'].str.contains('NLP sample', regex=False)].iloc[0]
row_f      = models_csv[models_csv['Model'].str.contains('Model F', regex=False)].iloc[0]

r2_full       = float(row_f['OLS R²'])
r2_restricted = float(row_a_star['OLS R²'])
n             = int(row_f['N (countries)'])
k_full        = int(row_f['Predictors used'])
k_restricted  = int(row_a_star['Predictors used'])
q             = k_full - k_restricted
df1, df2      = q, n - k_full - 1

delta_r2  = r2_full - r2_restricted
f_stat    = (delta_r2 / q) / ((1 - r2_full) / df2)
p_value   = float(1 - f_dist.cdf(f_stat, df1, df2))
partial_r2 = delta_r2 / (1 - r2_restricted)

print(f'Nested F-test: F({df1}, {df2}) = {f_stat:.3f}, p = {p_value:.4f}')
print(f'Delta R2 = {delta_r2:.3f}, Partial R2 = {partial_r2:.3f}')

def sig_label(p):
    if p < 0.01: return '***'
    if p < 0.05: return '**'
    if p < 0.10: return '*'
    return 'n.s.'

# ── Read individual variable results ──────────────────────────────────
def read_ols_coef(path, variable):
    try:
        with open(path, encoding='utf-8') as fh:
            for line in fh:
                stripped = line.strip()
                if stripped.startswith(variable):
                    parts = stripped.split()
                    if len(parts) >= 5:
                        return float(parts[1]), float(parts[4])
    except Exception:
        pass
    return None, None

avail_vars = [
    ('cereal_loss_pct',              'Post-harvest cereal loss'),
    ('trade_pct_gdp',                'Market integration / value-chain proxy'),
    ('rural_electricity_access_pct', 'Rural electricity for storage/processing'),
    ('fertiliser_efficiency',        'Fertiliser efficiency'),
    ('food_price_inflation_pct',     'Food price inflation'),
]

ols_f_path = 'outputs/tables/ols_Model_F_—_NLP-Discovered_Themes.txt'
# Try both naming conventions
if not os.path.exists(ols_f_path):
    for fname in os.listdir('outputs/tables'):
        if 'Model_F' in fname and fname.endswith('.txt'):
            ols_f_path = os.path.join('outputs/tables', fname)
            break

coef_rows = []
for var, theme in avail_vars:
    coef, pval = read_ols_coef(ols_f_path, var)
    coef_rows.append({
        'theme': theme, 'proxy_variable': var,
        'coefficient': coef, 'p_value': pval,
        'significance': sig_label(pval) if pval is not None else 'n/a',
    })
synth_df = pd.DataFrame(coef_rows)
synth_df.to_csv('outputs/tables/nlp_empirical_synthesis.csv', index=False)
print(synth_df[['theme','coefficient','p_value','significance']].to_string(index=False))


In [ ]:
# ── Bootstrap boot note ───────────────────────────────────────────────
boot_note_lines = []
n_boot_sig = 0
if os.path.exists('outputs/tables/bootstrap_confidence_intervals.csv'):
    boot = pd.read_csv('outputs/tables/bootstrap_confidence_intervals.csv')
    boot_f = boot[boot['model'] == 'Model F'].copy()
    for var, _ in avail_vars:
        row = boot_f[boot_f['variable'] == var]
        if len(row) == 0:
            continue
        lo = float(row['ci_lower_95'].iloc[0])
        hi = float(row['ci_upper_95'].iloc[0])
        excludes = lo * hi > 0
        if excludes:
            n_boot_sig += 1
        status = 'excludes zero' if excludes else 'crosses zero'
        boot_note_lines.append(f'  - {var}: 95% CI {status} [{lo:.3f}, {hi:.3f}]')

# ── Build narrative ────────────────────────────────────────────────────
adj_a = float(row_a_star['OLS Adj R²'])
adj_f = float(row_f['OLS Adj R²'])
adj_dir = 'increases' if adj_f >= adj_a else 'decreases'

if p_value < 0.05:
    conclusion = (
        f'The result is {sig_label(p_value)}: the NLP block adds '
        f'{partial_r2:.1%} partial R2 and this IS statistically '
        f'significant (p = {p_value:.3f}). '
        f'Adjusted R2 {adj_dir} from {adj_a:.3f} to {adj_f:.3f}, '
        f'confirming genuine explanatory power.'
    )
else:
    conclusion = (
        f'The result is {sig_label(p_value)}: the NLP block adds '
        f'{partial_r2:.1%} partial R2 but this is not statistically '
        f'distinguishable from noise (p = {p_value:.3f}). '
        f'Adjusted R2 {adj_dir} from {adj_a:.3f} to {adj_f:.3f}.'
    )

narrative_parts = [
    'PHASE F SYNTHESIS',
    f'Generated: {datetime.date.today().isoformat()}',
    '',
    '1. Nested F-test',
    f'   Model A* vs Model F (same N={n} countries)',
    f'   Delta R2      = {delta_r2:.3f}',
    f'   Partial R2    = {partial_r2:.3f}  ({partial_r2:.1%} of unexplained variance)',
    f'   F({df1},{df2}) = {f_stat:.3f},  p = {p_value:.4f}  {sig_label(p_value)}',
    f'   Adj R2 change = {adj_a:.3f} -> {adj_f:.3f}',
    '',
    '2. Conclusion',
    f'   {conclusion}',
    '',
    '3. Individual variable results (Model F)',
    synth_df.to_string(index=False),
    '',
    '4. Bootstrap 95% CIs',
    *boot_note_lines,
]
narrative = '\n'.join(narrative_parts)
with open('outputs/narrative/phase_f_synthesis.txt', 'w', encoding='utf-8') as fh:
    fh.write(narrative)
print('Phase F synthesis written.')
print(narrative)


---
## Pipeline Complete

All outputs saved under `outputs/`:

| File | Contents |
|------|----------|
| `outputs/tables/model_comparison.csv` | Model A/B/C/F/A★ R² table |
| `outputs/tables/nlp_empirical_synthesis.csv` | NLP theme → OLS coefficient |
| `outputs/tables/bootstrap_confidence_intervals.csv` | 95% CIs for all predictors |
| `outputs/tables/ols_*.txt` | Full OLS tables (HC3 SE) |
| `outputs/figures/model_r2_comparison.png` | R² progression chart |
| `outputs/figures/model_f_nlp_coefficients.png` | Model F coefficient chart |
| `outputs/figures/lda_coherence_curve.png` | LDA coherence sweep |
| `outputs/narrative/phase_f_synthesis.txt` | Plain-text synthesis narrative |
| `data/processed/nmf_availability_topics.csv` | NMF topic keywords |
| `data/processed/tfidf_top_keywords.csv` | Corpus TF-IDF top terms |

> **Key result:** Nested F-test F(5,147) = 3.649, p = 0.004 (***).  
> Rural electricity access (`rural_electricity_access_pct`) is the NLP-discovered predictor with the clearest empirical support (OLS p=0.012, bootstrap CI excludes zero).